In [ ]:
# Which relay was the higher rival bid on?  Reads per-relay bid traces (ethPandaOps Xatu) for 24 sample days.  ~20-40 min, resumable: just run again.
import os, time, zipfile, requests, numpy as np, pandas as pd, pyarrow as pa, pyarrow.parquet as pq
from google.colab import drive
for _ in range(5):
    try: drive.mount('/content/drive', force_remount=True); break
    except Exception as e: print('mount retry', e); time.sleep(10)
OUT = '/content/drive/MyDrive/pbs_all_bids'; RES = OUT + '/xatu'; os.makedirs(RES, exist_ok=True)
DAYS = ['2024-10-15', '2024-11-15', '2024-12-15', '2025-01-15', '2025-02-15', '2025-03-15', '2025-04-15', '2025-05-15', '2025-06-15', '2025-07-16', '2025-08-15', '2025-09-15', '2025-10-15', '2025-11-15',
        '2025-12-15', '2026-01-15', '2026-02-15', '2026-03-15', '2026-04-15', '2026-05-06', '2026-05-13', '2026-05-20', '2026-05-27', '2025-11-25']
URL = 'https://data.ethpandaops.io/xatu/mainnet/databases/default/mev_relay_bid_trace/{y}/{m}/{d}.parquet'
dl = pd.read_parquet(f'{OUT}/delivered_slim.parquet'); dl['builder_pubkey'] = dl.builder_pubkey.astype(str); dl = dl.set_index('slot')
COLS = ['slot', 'relay_name', 'block_hash', 'builder_pubkey', 'value', 'timestamp_ms']
def to_float(col):
    if pa.types.is_fixed_size_binary(col.type) or pa.types.is_binary(col.type) or pa.types.is_large_binary(col.type):
        b = col.to_pylist(); return np.array([float(int.from_bytes(x, 'little')) if x is not None else np.nan for x in b])
    if pa.types.is_string(col.type) or pa.types.is_large_string(col.type): return pd.to_numeric(pd.Series(col.to_pylist()), errors='coerce').to_numpy('float64')
    if pa.types.is_decimal(col.type): return np.array([float(x) if x is not None else np.nan for x in col.to_pylist()])
    return col.to_numpy(zero_copy_only=False).astype('float64')
def as_str(col):
    v = col.to_pylist(); return [x.decode() if isinstance(x, (bytes, bytearray)) else x for x in v]
def fetch(day):
    y, m, d = day.split('-'); tried = []
    for mm, dd in [(str(int(m)), str(int(d))), (m, d)]:
        u = URL.format(y=y, m=mm, d=dd); tried.append(u)
        r = requests.get(u, stream=True, timeout=300)
        if r.status_code == 200:
            with open('/content/x.parquet', 'wb') as f:
                for ch in r.iter_content(16 << 20): f.write(ch)
            return os.path.getsize('/content/x.parquet')
    raise RuntimeError(f'not found: {tried} (last status {r.status_code})')
def one_day(day):
    out = f'{RES}/{day}.parquet'
    if os.path.exists(out): return 'done already'
    t0 = time.time(); size = fetch(day); f = pq.ParquetFile('/content/x.parquet')
    if day == DAYS[0]: print('schema:', {n: str(f.schema_arrow.field(n).type) for n in COLS if n in f.schema_arrow.names})
    slots = dl.index.to_numpy(); parts = []; rows = 0; PK = {}; RL = {}
    def codes(col, D):                                                                              # strings -> small integer codes, no Python objects kept
        d = col.dictionary_encode(); vals = as_str(d.dictionary); m = np.array([D.setdefault(v, len(D)) for v in vals], 'int32'); return m[d.indices.to_numpy(zero_copy_only=False)]
    for b in f.iter_batches(batch_size=1 << 19, columns=COLS):
        rows += b.num_rows; s = b.column('slot').to_numpy(zero_copy_only=False).astype('int64'); keep = np.flatnonzero(np.isin(s, slots))
        if not len(keep): continue
        b = b.take(pa.array(keep)); s = s[keep]; h = np.array([x[:18] for x in as_str(b.column('block_hash'))])
        p = pd.DataFrame({'slot': s.astype('int32'), 'relay': codes(b.column('relay_name'), RL).astype('int8'), 'pk': codes(b.column('builder_pubkey'), PK).astype('int16'), 'v': to_float(b.column('value')),
                          'ts': b.column('timestamp_ms').to_numpy(zero_copy_only=False).astype('float64'), 'win': h == dl.hash18.reindex(s).to_numpy()})
        parts.append(p.sort_values('ts').drop_duplicates(['slot', 'relay', 'pk', 'v', 'win'])); del b, h, p
    if not parts: print(day, 'no delivered slots in file', rows); return 'empty'
    x = pd.concat(parts, ignore_index=True); del parts
    x = x.sort_values('ts').drop_duplicates(['slot', 'relay', 'pk', 'v', 'win'])                    # several collectors report the same bid
    x['relay'] = pd.Categorical.from_codes(x.relay.to_numpy(), categories=list(RL)); x['pk'] = pd.Categorical.from_codes(x.pk.to_numpy(), categories=list(PK))
    x['t'] = (x.ts - (1606824023 + 12 * x.slot.astype('int64')) * 1000.0).astype('float32'); x['dv'] = dl.value_wei.reindex(x.slot).to_numpy('float64'); x = x.drop(columns='ts')
    x = x[x.v < 1e21]
    nkept = len(x); nslots = x.slot.nunique(); rel = x.groupby('relay', observed=True).size().sort_values(ascending=False)
    sr = x.groupby(['slot', 'relay'], observed=True).agg(n=('v', 'size'), nk=('pk', 'nunique'), vtop=('v', 'max')).reset_index()
    sr = sr.merge(x[x.win].groupby(['slot', 'relay'], observed=True).t.min().rename('tw').reset_index(), on=['slot', 'relay'], how='left')
    sr['twg'] = sr.groupby('slot').tw.transform('min'); sr['dv'] = dl.value_wei.reindex(sr.slot).to_numpy('float64')
    x = x.merge(sr[['slot', 'relay', 'tw', 'twg']], on=['slot', 'relay'], how='left')
    g = x.groupby(['slot', 'relay', 'pk'], observed=True); k = g.agg(n=('v', 'size'), vmax=('v', 'max'), dv=('dv', 'first')).reset_index()
    def live(cut, nm):                                                                               # each key's latest bid on this relay no later than the cut
        y = x[x.t <= x[cut]]; y = y.loc[y.groupby(['slot', 'relay', 'pk'], observed=True).t.idxmax(), ['slot', 'relay', 'pk', 'v', 't']]
        return y.rename(columns={'v': 'live_' + nm, 't': 'tlive_' + nm})
    k = k.merge(live('tw', 'w'), on=['slot', 'relay', 'pk'], how='left').merge(live('twg', 'g'), on=['slot', 'relay', 'pk'], how='left')
    y = x[x.t <= x.twg]; k = k.merge(y.groupby(['slot', 'relay', 'pk'], observed=True).v.max().rename('vmax_pre').reset_index(), on=['slot', 'relay', 'pk'], how='left')
    k['isw'] = k.pk.astype(str).to_numpy() == dl.builder_pubkey.reindex(k.slot).to_numpy()
    k = k[(k.vmax >= 0.9 * k.dv) | k.isw]
    for c in ['vmax', 'live_w', 'live_g', 'vmax_pre']: k[c] = (k[c] / k.dv).astype('float32')
    del x, y; k = k.drop(columns='dv'); k['day'] = day; sr['vtop'] = (sr.vtop / sr.dv).astype('float32'); sr = sr.drop(columns='dv'); sr['day'] = day
    k['relay'] = k.relay.astype(str); k['pk'] = k.pk.astype(str); sr['relay'] = sr.relay.astype(str)
    sr.to_parquet(f'{RES}/{day}_relays.parquet', index=False); k.to_parquet(out + '.tmp', index=False); os.replace(out + '.tmp', out)
    print(f'{day}: {size / 1e6:.0f} MB, {rows:,} rows, {nkept:,} kept, {nslots:,} slots, bids/slot/relay median {sr.n.median():.0f}, winner found on a relay in {sr.groupby("slot").tw.min().notna().mean():.1%} of slots, {time.time() - t0:.0f}s | ' + ', '.join(f'{r}:{n}' for r, n in rel.items()), flush=True)
    return 'ok'
bad = 0
for i, day in enumerate(DAYS):
    try: one_day(day)
    except Exception as e:
        bad += 1; print(f'{day}: FAILED {type(e).__name__}: {e}', flush=True)
        if i == 0 or bad >= 4: raise
fs = sorted(f for f in os.listdir(RES) if f.endswith('.parquet'))
out = f'{OUT}/xatu_for_claude.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_STORED) as z:
    for f in fs: z.write(f'{RES}/{f}', f)
print(f'DONE: {len(fs)} files, {os.path.getsize(out) / 1e6:.0f} MB -> MyDrive/pbs_all_bids/xatu_for_claude.zip')
